# Smart Expense Classifier - EDA & Preprocessing
## VioletPro Internship Task

**Intern:** [Your Name]  
**Date:** November 12, 2026  
**Days:** 1-4 (Dataset, EDA, Preprocessing)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/sample_transactions.csv')

print(f'Dataset Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('Dataset Info:')
print(f'Total Transactions: {len(df):,}')
print(f'Features: {df.shape[1]}')
print(f'\nMissing Values:')
print(df.isnull().sum())
print(f'\nData Types:')
print(df.dtypes)

### 3.1 Category Distribution

In [ ]:
# Category distribution
category_counts = df['category'].value_counts()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
category_counts.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Category Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%')
axes[1].set_title('Category Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nCategory Statistics:')
print(category_counts)

### 3.2 Text Length Analysis

In [ ]:
# Text length analysis
df['text_length'] = df['description'].str.len()
df['word_count'] = df['description'].str.split().str.len()

print('Text Length Statistics:')
print(df[['text_length', 'word_count']].describe())

# Plot distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Character length
df['text_length'].hist(bins=30, ax=axes[0], color='coral')
axes[0].set_title('Character Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')

# Word count
df['word_count'].hist(bins=20, ax=axes[1], color='lightgreen')
axes[1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 3.3 Common Words per Category

In [ ]:
# Function to get top words per category
def get_top_words(texts, n=10):
    words = ' '.join(texts).lower().split()
    return Counter(words).most_common(n)

print('Top 10 Words per Category:\n')
for category in df['category'].unique():
    cat_texts = df[df['category'] == category]['description']
    top_words = get_top_words(cat_texts)
    print(f'\n{category}:')
    for word, count in top_words:
        print(f'  {word}: {count}')

## 4. Text Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
import nltk

# Download stopwords if not already done
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess_text(text):
    """
    Preprocess text:
    - Lowercase
    - Remove punctuation
    - Remove numbers
    - Remove stop words
    """
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove extra spaces
    text = ' '.join(text.split())
    
    # Remove stop words
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    
    return ' '.join(words)

# Apply preprocessing
df['processed_text'] = df['description'].apply(preprocess_text)

# Show examples
print('Preprocessing Examples:\n')
for i in range(5):
    print(f'Original: {df.iloc[i]["description"]}')
    print(f'Processed: {df.iloc[i]["processed_text"]}')
    print()

## 5. Save Processed Data

In [ ]:
# Save processed dataset
df.to_csv('../data/processed_transactions.csv', index=False)
print('✅ Processed data saved to: data/processed_transactions.csv')
print(f'\nFinal dataset shape: {df.shape}')

## 6. Summary

### Key Findings:
- Dataset contains 1,500 transactions across 9 categories
- Categories are well-balanced
- Average text length: ~20-30 characters
- Common words identified per category
- Text preprocessing applied successfully

### Next Steps:
- Split data into train/validation/test sets
- Build baseline model (TF-IDF + Logistic Regression)
- Evaluate performance